# Assignment 3 — MM-IMDb M2

Unimodal image baseline, unimodal text baseline, and a simple fusion baseline.

- Image: ResNet-18, ImageNet weights, fine-tuned.
- Text: DistilBERT-base-uncased, fine-tuned.
- Simple fusion: frozen image and text encoders, cached features, concatenation head.
- Metrics: macro-F1, micro-F1, sample-wise F1, exact match.

[TODO] GMU, modality dropout, ablation, missing-modality tests, and error analysis remain for M3.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "assignment3").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "assignment3" / "src"))

import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer

from utils.mmimdb import (
    build_label_matrix,
    build_splits,
    get_genres,
    load_mmimdb,
    make_image_loader,
    make_text_loader,
)
from models import (
    TEXT_MODEL_NAME,
    build_fusion_head,
    build_image_encoder,
    build_image_model,
    build_text_encoder,
    build_text_model,
)
from training_loop import (
    extract_image_features,
    extract_text_features,
    fit_fusion,
    fit_image,
    fit_text,
    metrics_from_probabilities,
    run_feature_epoch,
    run_image_epoch,
    run_text_epoch,
)

SEED = 42
IMAGE_EPOCHS = 10
TEXT_EPOCHS = 5
FUSION_EPOCHS = 30
IMAGE_BATCH = 256
TEXT_BATCH = 64
FUSION_BATCH = 256
WORKERS = 8
LIMIT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

RESULTS_DIR = ROOT / "assignment3" / "results"
CHECKPOINT_DIR = ROOT / "assignment3" / "checkpoints"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)

device: cuda


In [2]:
dataset = load_mmimdb()
genres = get_genres(dataset)
labels = build_label_matrix(dataset, genres)
splits = build_splits(dataset, genres)

if LIMIT:
    splits = {name: indices[:LIMIT] for name, indices in splits.items()}

print("genres:", len(genres))
print("splits:", {name: len(indices) for name, indices in splits.items()})
counts = labels.sum(axis=1)
print("labels per movie: mean", float(counts.mean()), "min", int(counts.min()), "max", int(counts.max()))


genres: 23
splits: {'train': 15553, 'dev': 2595, 'test': 7805}
labels per movie: mean 2.4846489429473877 min 1 max 10


In [3]:
image_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
image_model = build_image_model(len(genres), backbone="resnet18").to(DEVICE)
image_history, image_best_epoch, image_thresholds = fit_image(
    image_model,
    image_loaders,
    DEVICE,
    epochs=IMAGE_EPOCHS,
    lr=3e-4,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "image_resnet18.pt",
)

criterion = nn.BCEWithLogitsLoss()
_, image_test_probs, image_test_targets = run_image_epoch(
    image_model, image_loaders["test"], criterion, DEVICE, AMP
)
image_metrics = metrics_from_probabilities(
    image_test_probs, image_test_targets, image_thresholds
)
print({key: image_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/10 | train loss 0.3225 | val loss 0.2592 macro-F1 0.3104 | 82.6s


epoch 02/10 | train loss 0.2378 | val loss 0.2579 macro-F1 0.3312 | 59.7s


epoch 03/10 | train loss 0.2162 | val loss 0.2633 macro-F1 0.3265 | 15.5s


epoch 04/10 | train loss 0.1893 | val loss 0.2740 macro-F1 0.3279 | 52.8s


epoch 05/10 | train loss 0.1606 | val loss 0.2867 macro-F1 0.3274 | 40.7s


epoch 06/10 | train loss 0.1343 | val loss 0.3094 macro-F1 0.3108 | 46.7s


epoch 07/10 | train loss 0.1094 | val loss 0.3131 macro-F1 0.3071 | 17.8s


epoch 08/10 | train loss 0.0900 | val loss 0.3320 macro-F1 0.3063 | 11.5s


epoch 09/10 | train loss 0.0725 | val loss 0.3409 macro-F1 0.3133 | 14.4s


epoch 10/10 | train loss 0.0617 | val loss 0.3433 macro-F1 0.3072 | 13.7s


{'macro_f1': 0.3159713622045906, 'micro_f1': 0.4312789772837494, 'samples_f1': 0.4271024611251464, 'exact_match': 0.022549647661755285}


In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
text_model = build_text_model(len(genres), model_name="distilbert-base-uncased").to(DEVICE)
text_history, text_best_epoch, text_thresholds = fit_text(
    text_model,
    text_loaders,
    DEVICE,
    epochs=TEXT_EPOCHS,
    lr=2e-5,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "text_distilbert.pt",
)

_, text_test_probs, text_test_targets = run_text_epoch(
    text_model, text_loaders["test"], criterion, DEVICE, AMP
)
text_metrics = metrics_from_probabilities(
    text_test_probs, text_test_targets, text_thresholds
)
print({key: text_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch 01/5 | train loss 0.3331 | val loss 0.2472 macro-F1 0.3679 | 21.1s


epoch 02/5 | train loss 0.2203 | val loss 0.2071 macro-F1 0.5029 | 14.1s


epoch 03/5 | train loss 0.1880 | val loss 0.1943 macro-F1 0.5483 | 14.9s


epoch 04/5 | train loss 0.1677 | val loss 0.1950 macro-F1 0.5665 | 14.6s


epoch 05/5 | train loss 0.1520 | val loss 0.1866 macro-F1 0.5816 | 15.3s


{'macro_f1': 0.5677920612870081, 'micro_f1': 0.6437311838399534, 'samples_f1': 0.6461750990130234, 'exact_match': 0.17245355541319668}


In [5]:
image_encoder = build_image_encoder(backbone="resnet18").to(DEVICE)
text_encoder = build_text_encoder(model_name="distilbert-base-uncased").to(DEVICE)

image_feature_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, False, WORKERS)
    for name, indices in splits.items()
}
text_feature_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, False, WORKERS)
    for name, indices in splits.items()
}

fusion_features = {}
fusion_labels = {}
for name in splits:
    image_features, image_targets = extract_image_features(
        image_encoder, image_feature_loaders[name], DEVICE, AMP
    )
    text_features, text_targets = extract_text_features(
        text_encoder, text_feature_loaders[name], DEVICE, AMP
    )
    assert torch.equal(image_targets, text_targets)
    fusion_features[name] = torch.cat([image_features, text_features], dim=1)
    fusion_labels[name] = image_targets

fusion_head = build_fusion_head(
    input_dim=fusion_features["train"].shape[1],
    hidden_dim=512,
    num_labels=len(genres),
).to(DEVICE)
fusion_history, fusion_best_epoch, fusion_thresholds = fit_fusion(
    fusion_head,
    fusion_features["train"],
    fusion_labels["train"],
    fusion_features["dev"],
    fusion_labels["dev"],
    DEVICE,
    epochs=FUSION_EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=FUSION_BATCH,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "fusion_mlp.pt",
)

dev_loader = DataLoader(
    TensorDataset(fusion_features["dev"], fusion_labels["dev"]),
    batch_size=FUSION_BATCH,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(fusion_features["test"], fusion_labels["test"]),
    batch_size=FUSION_BATCH,
    shuffle=False,
)
_, fusion_test_probs, fusion_test_targets = run_feature_epoch(
    fusion_head, test_loader, criterion, DEVICE
)
fusion_metrics = metrics_from_probabilities(
    fusion_test_probs, fusion_test_targets, fusion_thresholds
)
print({key: fusion_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/30 | train loss 0.2780 | val macro-F1 0.4298


epoch 05/30 | train loss 0.1847 | val macro-F1 0.5674


epoch 10/30 | train loss 0.1715 | val macro-F1 0.5879


epoch 15/30 | train loss 0.1662 | val macro-F1 0.5902


epoch 20/30 | train loss 0.1607 | val macro-F1 0.5931


epoch 25/30 | train loss 0.1562 | val macro-F1 0.5918


epoch 30/30 | train loss 0.1520 | val macro-F1 0.5913


{'macro_f1': 0.5823441913084881, 'micro_f1': 0.6450236171386098, 'samples_f1': 0.6379011599831588, 'exact_match': 0.1455477258167841}


In [6]:
rows = []
for name, model, metrics, history, best_epoch, thresholds in [
    ("Image-only ResNet-18", image_model, image_metrics, image_history, image_best_epoch, image_thresholds),
    ("Text-only DistilBERT", text_model, text_metrics, text_history, text_best_epoch, text_thresholds),
    ("Simple fusion", fusion_head, fusion_metrics, fusion_history, fusion_best_epoch, fusion_thresholds),
]:
    rows.append(
        {
            "model": name,
            "params": sum(p.numel() for p in model.parameters()),
            "best_epoch": int(best_epoch),
            "train_time_s": float(sum(history["epoch_time"])),
            "test_macro_f1": metrics["macro_f1"],
            "test_micro_f1": metrics["micro_f1"],
            "test_samples_f1": metrics["samples_f1"],
            "test_exact_match": metrics["exact_match"],
        }
    )

results_df = pd.DataFrame(rows)
results_df

,model,params,best_epoch,train_time_s,test_macro_f1,test_micro_f1,test_samples_f1,test_exact_match
0,Image-only ResNet-18,11188311,2,355.205640,0.315971,0.431279,0.427102,0.022550
1,Text-only DistilBERT,66971159,5,80.023185,0.567792,0.643731,0.646175,0.172454
2,Simple fusion,667671,24,64.651731,0.582344,0.645024,0.637901,0.145548


In [7]:
config = {
    "dataset": "MM-IMDb",
    "task": "23-label movie genre classification",
    "seed": SEED,
    "image_epochs": IMAGE_EPOCHS,
    "text_epochs": TEXT_EPOCHS,
    "fusion_epochs": FUSION_EPOCHS,
    "image_batch": IMAGE_BATCH,
    "text_batch": TEXT_BATCH,
    "fusion_batch": FUSION_BATCH,
    "limit": LIMIT,
    "amp": AMP,
    "device": str(DEVICE),
}
payload = {
    "config": config,
    "genres": genres,
    "splits": {name: len(indices) for name, indices in splits.items()},
    "results": rows,
    "history": {
        "image": image_history,
        "text": text_history,
        "fusion": fusion_history,
    },
    "thresholds": {
        "image": image_thresholds.tolist(),
        "text": text_thresholds.tolist(),
        "fusion": fusion_thresholds.tolist(),
    },
    "per_class_f1": {
        "image": image_metrics["per_class_f1"],
        "text": text_metrics["per_class_f1"],
        "fusion": fusion_metrics["per_class_f1"],
    },
}
with open(RESULTS_DIR / "m2_results.json", "w") as handle:
    json.dump(payload, handle, indent=2)
results_df.to_csv(RESULTS_DIR / "m2_results.csv", index=False)
print("saved", RESULTS_DIR / "m2_results.json")

saved /raid/home/an.nv/main/deep-learning/CO3133-DeepLearning/assignment3/results/m2_results.json
